<a href="https://colab.research.google.com/github/piyush12kunjilwar/CUDA-Kernel-Optimization/blob/main/cuda_kernel_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# Module 02: CUDA Kernel Optimization
# Author: Piyush Kunjilwar
# Goal: Write, profile and optimize CUDA kernels
# from naive → shared memory → Triton → Flash Attention
# ============================================

!nvidia-smi
!nvcc --version

In [ ]:
import torch
import time
import numpy as np
import matplotlib.pyplot as plt

print("=" * 55)
print("🖥️  CUDA Environment")
print("=" * 55)

device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)

print(f"GPU Name:                {props.name}")
print(f"Total Memory:            {props.total_memory / 1e9:.1f} GB")
print(f"CUDA Capability:         {props.major}.{props.minor}")
print(f"Streaming Multiprocessors: {props.multi_processor_count}")
print(f"Max threads per block:   {props.max_threads_per_block}")
print(f"Max threads per SM:      {props.max_threads_per_multi_processor}")
print(f"Warp size:               32")
print(f"CUDA version:            {torch.version.cuda}")
print(f"PyTorch version:         {torch.__version__}")

# Calculate theoretical peak performance
# Each SM on L4 has 128 CUDA cores
cuda_cores = props.multi_processor_count * 128
clock_mhz = props.clock_rate / 1000
peak_tflops = (cuda_cores * 2 * clock_mhz * 1e6) / 1e12

print(f"\n📊 Compute Analysis:")
print(f"Estimated CUDA cores:    {cuda_cores}")
print(f"Clock rate:              {clock_mhz:.0f} MHz")
print(f"Peak FP32 TFLOPS:        ~{peak_tflops:.1f}")
print(f"Max warps per SM:        {props.max_threads_per_multi_processor // 32}")
print("=" * 55)
print("\n💡 Key concepts:")
print("  Warp = 32 threads that execute in lockstep")
print("  SM = Streaming Multiprocessor (like a CPU core)")
print("  All threads in warp must follow same code path")
print("  Memory coalescing = adjacent threads access adjacent memory")

In [ ]:
!pip install -q triton ninja
import triton
print(f"✅ Triton version: {triton.__version__}")
print("✅ All tools ready — let's write kernels!")

In [ ]:
from torch.utils.cpp_extension import load_inline

# ============================================
# KERNEL 1: Naive Vector Addition
# This IS real CUDA C++ code
# We're writing actual GPU instructions
# ============================================

cuda_source = """
#include <torch/extension.h>
#include <cuda_runtime.h>

// __global__ = this function runs ON the GPU
// called FROM the CPU
// Every single thread runs this simultaneously
__global__ void vector_add_kernel(
    const float* a,
    const float* b,
    float* c,
    int n
) {
    // Each thread figures out its unique position
    // This is the most important line in CUDA
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Guard: don't go out of bounds
    if (idx < n) {
        c[idx] = a[idx] + b[idx];
    }
}

torch::Tensor vector_add_cuda(
    torch::Tensor a,
    torch::Tensor b
) {
    int n = a.size(0);
    auto c = torch::zeros_like(a);

    // 256 threads per block — standard choice
    // Must be multiple of 32 (warp size)
    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    // <<< >>> is CUDA kernel launch syntax
    // This launches thousands of threads simultaneously
    vector_add_kernel<<<blocks, threads>>>(
        a.data_ptr<float>(),
        b.data_ptr<float>(),
        c.data_ptr<float>(),
        n
    );

    cudaDeviceSynchronize();
    return c;
}
"""

cpp_source = """
torch::Tensor vector_add_cuda(torch::Tensor a, torch::Tensor b);
"""

print("⚙️  Compiling CUDA kernel...")
print("First compile ~30 seconds — cached after that")

vector_add = load_inline(
    name="vector_add",
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=["vector_add_cuda"],
    verbose=False,
    extra_cuda_cflags=["-O3"]
)

print("✅ Kernel compiled!")
print("\n💡 What just happened:")
print("  1. We wrote CUDA C++ code")
print("  2. nvcc compiled it to GPU machine code")
print("  3. Python can now call it directly")
print("  4. When called — 7,424 CUDA cores activate")

In [ ]:
print("🧪 Testing our kernel...")

n = 1_000_000
a = torch.randn(n, device='cuda', dtype=torch.float32)
b = torch.randn(n, device='cuda', dtype=torch.float32)

# Our kernel
c_ours = vector_add.vector_add_cuda(a, b)

# PyTorch reference
c_pytorch = a + b

# Verify correctness
max_diff = torch.max(torch.abs(c_ours - c_pytorch)).item()
print(f"Max difference vs PyTorch: {max_diff:.2e}")

if max_diff < 1e-5:
    print("✅ Correct! Our kernel matches PyTorch")
else:
    print("❌ Bug in kernel!")

print(f"\n📊 What just happened on the GPU:")
print(f"   Elements processed:  {n:,}")
print(f"   Threads launched:    {((n+255)//256)*256:,}")
print(f"   Thread blocks:       {(n+255)//256:,}")
print(f"   Threads per block:   256")
print(f"   Warps launched:      {((n+255)//256)*8:,}")

In [ ]:
def benchmark(fn, *args, runs=1000, warmup=100):
    for _ in range(warmup):
        fn(*args)
    torch.cuda.synchronize()

    times = []
    for _ in range(runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn(*args)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1e6)

    return {
        "mean_us": np.mean(times),
        "p50_us":  np.percentile(times, 50),
        "p95_us":  np.percentile(times, 95),
        "min_us":  np.min(times),
    }

print("=" * 55)
print("📊 Vector Addition Benchmark (1M elements)")
print("=" * 55)

our_results     = benchmark(vector_add.vector_add_cuda, a, b)
pytorch_results = benchmark(torch.add, a, b)

print("\n📈 Our naive CUDA kernel:")
for k, v in our_results.items():
    print(f"   {k}: {v:.2f} μs")

print("\n📈 PyTorch built-in:")
for k, v in pytorch_results.items():
    print(f"   {k}: {v:.2f} μs")

ratio = our_results['mean_us'] / pytorch_results['mean_us']
print(f"\n📊 Ratio: {ratio:.2f}x")

if ratio > 1:
    print(f"   PyTorch is {ratio:.2f}x faster — expected!")
    print(f"   PyTorch uses cuBLAS — decades of optimization")
    print(f"   Our kernel teaches the fundamentals")
    print(f"   Next: shared memory will close this gap")
else:
    print(f"   Our kernel is {1/ratio:.2f}x faster — impressive!")

# Store for later comparison
results = {
    "naive_cuda": our_results,
    "pytorch":    pytorch_results
}

In [ ]:
# ============================================
# KERNEL 2: Optimized Vector Addition
# Improvements:
# 1. Grid-stride loop — handles any size efficiently
# 2. __restrict__ hints — tells compiler no aliasing
# 3. Pragma unroll — loop unrolling
# 4. Better launch config
# ============================================

cuda_source_v2 = """
#include <torch/extension.h>
#include <cuda_runtime.h>

// __restrict__ tells compiler these pointers don't overlap
// Allows more aggressive optimization
__global__ void vector_add_v2_kernel(
    const float* __restrict__ a,
    const float* __restrict__ b,
    float* __restrict__ c,
    int n
) {
    // Grid-stride loop pattern
    // Instead of one element per thread
    // each thread handles MULTIPLE elements
    // This improves memory access patterns
    // and handles arrays larger than our grid
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    // Each thread processes multiple elements
    // strided across the array
    for (int i = idx; i < n; i += stride) {
        c[i] = a[i] + b[i];
    }
}

// Vectorized version — loads 4 floats at once (float4)
// This is how production kernels work
// Instead of reading 1 float (4 bytes) per memory transaction
// we read 4 floats (16 bytes) — 4x better bandwidth utilization
__global__ void vector_add_v3_kernel(
    const float4* __restrict__ a,
    const float4* __restrict__ b,
    float4* __restrict__ c,
    int n  // n is number of float4 elements
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = idx; i < n; i += stride) {
        float4 a_val = a[i];
        float4 b_val = b[i];
        // Process 4 floats simultaneously
        c[i] = make_float4(
            a_val.x + b_val.x,
            a_val.y + b_val.y,
            a_val.z + b_val.z,
            a_val.w + b_val.w
        );
    }
}

torch::Tensor vector_add_v2_cuda(
    torch::Tensor a,
    torch::Tensor b
) {
    int n = a.size(0);
    auto c = torch::zeros_like(a);

    // Better launch config
    // Use more threads per block — better SM utilization
    int threads = 512;
    // Limit blocks to SM count * 4 for grid-stride
    int blocks = min(
        (n + threads - 1) / threads,
        58 * 4  // 58 SMs * 4 blocks per SM
    );

    vector_add_v2_kernel<<<blocks, threads>>>(
        a.data_ptr<float>(),
        b.data_ptr<float>(),
        c.data_ptr<float>(),
        n
    );

    return c;
}

torch::Tensor vector_add_v3_cuda(
    torch::Tensor a,
    torch::Tensor b
) {
    int n = a.size(0);
    auto c = torch::zeros_like(a);

    // n/4 because each thread handles float4
    int n4 = n / 4;
    int threads = 512;
    int blocks = min(
        (n4 + threads - 1) / threads,
        58 * 4
    );

    vector_add_v3_kernel<<<blocks, threads>>>(
        reinterpret_cast<float4*>(a.data_ptr<float>()),
        reinterpret_cast<float4*>(b.data_ptr<float>()),
        reinterpret_cast<float4*>(c.data_ptr<float>()),
        n4
    );

    return c;
}
"""

cpp_source_v2 = """
torch::Tensor vector_add_v2_cuda(torch::Tensor a, torch::Tensor b);
torch::Tensor vector_add_v3_cuda(torch::Tensor a, torch::Tensor b);
"""

print("⚙️  Compiling optimized kernels...")

vector_add_v2 = load_inline(
    name="vector_add_v2",
    cpp_sources=cpp_source_v2,
    cuda_sources=cuda_source_v2,
    functions=["vector_add_v2_cuda", "vector_add_v3_cuda"],
    verbose=False,
    extra_cuda_cflags=["-O3", "--use_fast_math"]
)

print("✅ Optimized kernels compiled!")
print("\n💡 What we improved:")
print("  v2: Grid-stride loop + __restrict__ + better launch config")
print("  v3: float4 vectorization — 4x memory bandwidth efficiency")

In [ ]:
# Verify correctness first
c_v2 = vector_add_v2.vector_add_v2_cuda(a, b)
c_v3 = vector_add_v2.vector_add_v3_cuda(a, b)

diff_v2 = torch.max(torch.abs(c_v2 - c_pytorch)).item()
diff_v3 = torch.max(torch.abs(c_v3 - c_pytorch)).item()

print(f"v2 correctness: {diff_v2:.2e} {'✅' if diff_v2 < 1e-5 else '❌'}")
print(f"v3 correctness: {diff_v3:.2e} {'✅' if diff_v3 < 1e-5 else '❌'}")

# Benchmark all versions
print("\n" + "=" * 55)
print("📊 Full Benchmark — All Versions")
print(f"   Vector size: {n:,} elements")
print("=" * 55)

v2_results = benchmark(vector_add_v2.vector_add_v2_cuda, a, b)
v3_results = benchmark(vector_add_v2.vector_add_v3_cuda, a, b)

all_results = {
    "PyTorch built-in":      pytorch_results['mean_us'],
    "Naive CUDA (v1)":       our_results['mean_us'],
    "Grid-stride (v2)":      v2_results['mean_us'],
    "Float4 vectorized (v3)": v3_results['mean_us'],
}

print(f"\n{'Method':<25} {'Mean (μs)':>10} {'vs PyTorch':>12}")
print("-" * 50)
for name, mean in all_results.items():
    ratio = mean / pytorch_results['mean_us']
    symbol = "✅" if ratio <= 1.1 else "📈"
    print(f"{name:<25} {mean:>10.2f} {ratio:>10.2f}x {symbol}")

# Store results
results["grid_stride_v2"] = v2_results
results["float4_v3"] = v3_results

print(f"\n💡 Key insight:")
print(f"   float4 reads 16 bytes per transaction vs 4 bytes")
print(f"   Same compute, 4x better memory bandwidth utilization")
print(f"   This is memory coalescing in practice")

In [ ]:
# ============================================
# KERNEL 3: Matrix Multiplication
# THIS is what matters for ML
# Every linear layer, attention, everything
# reduces to matrix multiply
# ============================================

cuda_matmul = """
#include <torch/extension.h>
#include <cuda_runtime.h>

// NAIVE matrix multiply
// C = A × B
// A: (M x K), B: (K x N), C: (M x N)
//
// Problem: EVERY thread reads an ENTIRE row of A
// and ENTIRE column of B from HBM
// M*N*K total HBM reads — extremely memory bound
__global__ void matmul_naive_kernel(
    const float* __restrict__ A,
    const float* __restrict__ B,
    float* __restrict__ C,
    int M, int N, int K
) {
    // Each thread computes ONE element of C
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < N) {
        float sum = 0.0f;

        // Dot product of row from A and col from B
        // Each iteration reads from HBM — SLOW!
        for (int k = 0; k < K; k++) {
            sum += A[row * K + k] * B[k * N + col];
        }

        C[row * N + col] = sum;
    }
}

torch::Tensor matmul_naive_cuda(
    torch::Tensor A,
    torch::Tensor B
) {
    int M = A.size(0);
    int K = A.size(1);
    int N = B.size(1);

    auto C = torch::zeros({M, N}, A.options());

    // 2D thread blocks for 2D output matrix
    dim3 threads(16, 16);  // 256 threads per block
    dim3 blocks(
        (N + threads.x - 1) / threads.x,
        (M + threads.y - 1) / threads.y
    );

    matmul_naive_kernel<<<blocks, threads>>>(
        A

In [ ]:
from torch.utils.cpp_extension import load_inline

cuda_source_v2 = """
#include <torch/extension.h>
#include <cuda_runtime.h>

__global__ void vector_add_v2_kernel(
    const float* __restrict__ a,
    const float* __restrict__ b,
    float* __restrict__ c,
    int n
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = idx; i < n; i += stride) {
        c[i] = a[i] + b[i];
    }
}

__global__ void vector_add_v3_kernel(
    const float4* __restrict__ a,
    const float4* __restrict__ b,
    float4* __restrict__ c,
    int n
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = idx; i < n; i += stride) {
        float4 a_val = a[i];
        float4 b_val = b[i];
        c[i] = make_float4(
            a_val.x + b_val.x,
            a_val.y + b_val.y,
            a_val.z + b_val.z,
            a_val.w + b_val.w
        );
    }
}

torch::Tensor vector_add_v2_cuda(
    torch::Tensor a, torch::Tensor b
) {
    int n = a.size(0);
    auto c = torch::zeros_like(a);
    int threads = 512;
    int blocks = min((n + threads - 1) / threads, 58 * 4);
    vector_add_v2_kernel<<<blocks, threads>>>(
        a.data_ptr<float>(),
        b.data_ptr<float>(),
        c.data_ptr<float>(), n
    );
    return c;
}

torch::Tensor vector_add_v3_cuda(
    torch::Tensor a, torch::Tensor b
) {
    int n = a.size(0);
    auto c = torch::zeros_like(a);
    int n4 = n / 4;
    int threads = 512;
    int blocks = min((n4 + threads - 1) / threads, 58 * 4);
    vector_add_v3_kernel<<<blocks, threads>>>(
        reinterpret_cast<float4*>(a.data_ptr<float>()),
        reinterpret_cast<float4*>(b.data_ptr<float>()),
        reinterpret_cast<float4*>(c.data_ptr<float>()),
        n4
    );
    return c;
}
"""

cpp_source_v2 = """
torch::Tensor vector_add_v2_cuda(torch::Tensor a, torch::Tensor b);
torch::Tensor vector_add_v3_cuda(torch::Tensor a, torch::Tensor b);
"""

print("⚙️  Compiling optimized kernels...")
vector_add_v2 = load_inline(
    name="vector_add_v2",
    cpp_sources=cpp_source_v2,
    cuda_sources=cuda_source_v2,
    functions=["vector_add_v2_cuda", "vector_add_v3_cuda"],
    verbose=False,
    extra_cuda_cflags=["-O3", "--use_fast_math"]
)
print("✅ Compiled!")

In [ ]:
# Verify
c_v2 = vector_add_v2.vector_add_v2_cuda(a, b)
c_v3 = vector_add_v2.vector_add_v3_cuda(a, b)

diff_v2 = torch.max(torch.abs(c_v2 - c_pytorch)).item()
diff_v3 = torch.max(torch.abs(c_v3 - c_pytorch)).item()
print(f"v2 correct: {diff_v2:.2e} {'✅' if diff_v2 < 1e-5 else '❌'}")
print(f"v3 correct: {diff_v3:.2e} {'✅' if diff_v3 < 1e-5 else '❌'}")

# Benchmark
v2_results = benchmark(vector_add_v2.vector_add_v2_cuda, a, b)
v3_results = benchmark(vector_add_v2.vector_add_v3_cuda, a, b)

print("\n" + "=" * 55)
print("📊 All Versions Compared")
print("=" * 55)
print(f"\n{'Method':<25} {'Mean (μs)':>10} {'vs PyTorch':>12}")
print("-" * 50)

all_results = {
    "PyTorch built-in":       pytorch_results['mean_us'],
    "Naive CUDA v1":          our_results['mean_us'],
    "Grid-stride v2":         v2_results['mean_us'],
    "Float4 vectorized v3":   v3_results['mean_us'],
}

for name, mean in all_results.items():
    ratio = mean / pytorch_results['mean_us']
    symbol = "✅" if ratio <= 1.2 else "📈"
    print(f"{name:<25} {mean:>10.2f} {ratio:>10.2f}x {symbol}")

results["grid_stride_v2"] = v2_results
results["float4_v3"] = v3_results


In [ ]:
print("✅ Optimized kernels compiled!")
print("\n💡 What we improved:")
print("  v2: Grid-stride loop + __restrict__ + better launch config")
print("  v3: float4 vectorization — 4x memory bandwidth efficiency")

In [ ]:
print(f"   float4 reads 16 bytes per transaction vs 4 bytes")
print(f"   Same compute, 4x better memory bandwidth utilization")
print(f"   This is memory coalescing in practice")

In [ ]:
v2_results = benchmark(vector_add_v2.vector_add_v2_cuda, a, b)
v3_results = benchmark(vector_add_v2.vector_add_v3_cuda, a, b)

print(f"\n{'Method':<25} {'Mean (μs)':>10} {'vs PyTorch':>12}")
print("-" * 50)

all_results = {
    "PyTorch built-in":       pytorch_results['mean_us'],
    "Naive CUDA (v1)":        our_results['mean_us'],
    "Grid-stride (v2)":       v2_results['mean_us'],
    "Float4 vectorized (v3)": v3_results['mean_us'],
}

for name, mean in all_results.items():
    ratio = mean / pytorch_results['mean_us']
    symbol = "✅" if ratio <= 1.1 else "📈"
    print(f"{name:<25} {mean:>10.2f} {ratio:>10.2f}x {symbol}")

In [ ]:
# ============================================
# KERNEL 3: Matrix Multiplication
# This is THE most important kernel in ML
# Every transformer layer is matrix multiply
# We'll go from naive → tiled shared memory
# ============================================

cuda_matmul = """
#include <torch/extension.h>
#include <cuda_runtime.h>

// ============================================
// VERSION 1: Naive Matrix Multiply
// Every thread computes one output element
// C[row][col] = sum(A[row][k] * B[k][col])
// Problem: each thread reads entire row of A
// and entire col of B from slow HBM
// ============================================
__global__ void matmul_naive_kernel(
    const float* __restrict__ A,
    const float* __restrict__ B,
    float* __restrict__ C,
    int M, int N, int K
) {
    // Which output element am I computing?
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < N) {
        float sum = 0.0f;
        // This loop reads K elements from HBM
        // for EVERY thread — extremely inefficient
        for (int k = 0; k < K; k++) {
            sum += A[row * K + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

// ============================================
// VERSION 2: Tiled Matrix Multiply
// Uses SHARED MEMORY — the kitchen counter!
//
// Key idea: threads in a block COOPERATE
// to load a TILE of A and B into fast
// shared memory, then compute from there
//
// Instead of each thread reading from HBM K times
// the whole block reads ONE TILE together
// and reuses it many times
// ============================================

#define TILE_SIZE 16  // 16x16 tile = 256 threads per block

__global__ void matmul_tiled_kernel(
    const float* __restrict__ A,
    const float* __restrict__ B,
    float* __restrict__ C,
    int M, int N, int K
) {
    // Shared memory tiles — lives on SM, very fast!
    // __shared__ = stored in shared memory (not HBM)
    __shared__ float tile_A[TILE_SIZE][TILE_SIZE];
    __shared__ float tile_B[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;

    float sum = 0.0f;

    // Loop over tiles
    // Each iteration loads ONE TILE into shared memory
    // then ALL threads compute using that tile
    int num_tiles = (K + TILE_SIZE - 1) / TILE_SIZE;

    for (int t = 0; t < num_tiles; t++) {
        // COLLABORATIVE LOADING
        // Each thread loads ONE element of each tile
        // Together the block loads the whole tile
        int a_col = t * TILE_SIZE + threadIdx.x;
        int b_row = t * TILE_SIZE + threadIdx.y;

        // Load A tile with bounds checking
        if (row < M && a_col < K)
            tile_A[threadIdx.y][threadIdx.x] = A[row * K + a_col];
        else
            tile_A[threadIdx.y][threadIdx.x] = 0.0f;

        // Load B tile with bounds checking
        if (b_row < K && col < N)
            tile_B[threadIdx.y][threadIdx.x] = B[b_row * N + col];
        else
            tile_B[threadIdx.y][threadIdx.x] = 0.0f;

        // CRITICAL: Wait for ALL threads to finish loading
        // before any thread starts computing
        // Without this — race condition!
        __syncthreads();

In [ ]:
cuda_matmul = """
#include <torch/extension.h>
#include <cuda_runtime.h>

#define TILE_SIZE 16

__global__ void matmul_naive_kernel(
    const float* __restrict__ A,
    const float* __restrict__ B,
    float* __restrict__ C,
    int M, int N, int K
) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < K; k++) {
            sum += A[row * K + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

__global__ void matmul_tiled_kernel(
    const float* __restrict__ A,
    const float* __restrict__ B,
    float* __restrict__ C,
    int M, int N, int K
) {
    __shared__ float tile_A[TILE_SIZE][TILE_SIZE];
    __shared__ float tile_B[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;
    float sum = 0.0f;
    int num_tiles = (K + TILE_SIZE - 1) / TILE_SIZE;

    for (int t = 0; t < num_tiles; t++) {
        int a_col = t * TILE_SIZE + threadIdx.x;
        int b_row = t * TILE_SIZE + threadIdx.y;

        tile_A[threadIdx.y][threadIdx.x] =
            (row < M && a_col < K) ? A[row * K + a_col] : 0.0f;
        tile_B[threadIdx.y][threadIdx.x] =
            (b_row < K && col < N) ? B[b_row * N + col] : 0.0f;

        __syncthreads();

        #pragma unroll
        for (int k = 0; k < TILE_SIZE; k++) {
            sum += tile_A[threadIdx.y][k] * tile_B[k][threadIdx.x];
        }
        __syncthreads();
    }

    if (row < M && col < N) {
        C[row * N + col] = sum;
    }
}

torch::Tensor matmul_naive_cuda(torch::Tensor A, torch::Tensor B) {
    int M = A.size(0), K = A.size(1), N = B.size(1);
    auto C = torch::zeros({M, N}, A.options());
    dim3 threads(16, 16);
    dim3 blocks((N+15)/16, (M+15)/16);
    matmul_naive_kernel<<<blocks, threads>>>(
        A.data_ptr<float>(), B.data_ptr<float>(),
        C.data_ptr<float>(), M, N, K);
    cudaDeviceSynchronize();
    return C;
}

torch::Tensor matmul_tiled_cuda(torch::Tensor A, torch::Tensor B) {
    int M = A.size(0), K = A.size(1), N = B.size(1);
    auto C = torch::zeros({M, N}, A.options());
    dim3 threads(TILE_SIZE, TILE_SIZE);
    dim3 blocks((N+TILE_SIZE-1)/TILE_SIZE, (M+TILE_SIZE-1)/TILE_SIZE);
    matmul_tiled_kernel<<<blocks, threads>>>(
        A.data_ptr<float>(), B.data_ptr<float>(),
        C.data_ptr<float>(), M, N, K);
    cudaDeviceSynchronize();
    return C;
}
"""

print("✅ CUDA source defined!")

In [ ]:
from torch.utils.cpp_extension import load_inline

cpp_matmul = """
torch::Tensor matmul_naive_cuda(torch::Tensor A, torch::Tensor B);
torch::Tensor matmul_tiled_cuda(torch::Tensor A, torch::Tensor B);
"""

print("⚙️  Compiling matrix multiply kernels...")

matmul_module = load_inline(
    name="matmul_kernels",
    cpp_sources=cpp_matmul,
    cuda_sources=cuda_matmul,
    functions=["matmul_naive_cuda", "matmul_tiled_cuda"],
    verbose=False,
    extra_cuda_cflags=["-O3", "--use_fast_math"]
)

print("✅ Compiled!")
print("  Naive matmul: each thread reads from slow HBM")
print("  Tiled matmul: threads share fast shared memory")
print("  This is the core idea behind Flash Attention!")

In [ ]:
# Test matrices
M, N, K = 1024, 1024, 1024
A = torch.randn(M, K, device='cuda', dtype=torch.float32)
B = torch.randn(K, N, device='cuda', dtype=torch.float32)

# Verify correctness
C_naive  = matmul_module.matmul_naive_cuda(A, B)
C_tiled  = matmul_module.matmul_tiled_cuda(A, B)
C_pytorch = torch.mm(A, B)

diff_naive = torch.max(torch.abs(C_naive - C_pytorch)).item()
diff_tiled = torch.max(torch.abs(C_tiled - C_pytorch)).item()

print(f"Naive correctness:  {diff_naive:.2e} {'✅' if diff_naive < 0.01 else '❌'}")
print(f"Tiled correctness:  {diff_tiled:.2e} {'✅' if diff_tiled < 0.01 else '❌'}")

# Benchmark
print("\n" + "=" * 55)
print(f"📊 Matrix Multiply Benchmark ({M}x{K} @ {K}x{N})")
print("=" * 55)

naive_r   = benchmark(matmul_module.matmul_naive_cuda, A, B, runs=200)
tiled_r   = benchmark(matmul_module.matmul_tiled_cuda, A, B, runs=200)
pytorch_r = benchmark(torch.mm, A, B, runs=200)

all_r = {
    "PyTorch (cuBLAS)": pytorch_r['mean_us'],
    "Naive CUDA":       naive_r['mean_us'],
    "Tiled (shared mem)": tiled_r['mean_us'],
}

print(f"\n{'Method':<25} {'Mean (μs)':>10} {'vs PyTorch':>12} {'Speedup vs Naive':>18}")
print("-" * 70)
for name, mean in all_r.items():
    ratio_pt    = mean / pytorch_r['mean_us']
    ratio_naive = naive_r['mean_us'] / mean
    print(f"{name:<25} {mean:>10.1f} {ratio_pt:>10.2f}x {ratio_naive:>16.2f}x")

print(f"\n💡 Key insight:")
print(f"   Tiled version reduces HBM reads by {TILE_SIZE}x")
print(f"   Data loaded once into shared memory")
print(f"   Reused {TILE_SIZE} times before eviction")
print(f"   This is exactly what Flash Attention does!")

# Store
results['matmul_naive'] = naive_r
results['matmul_tiled'] = tiled_r
results['matmul_pytorch'] = pytorch_r

In [ ]:
TILE_SIZE = 16
print(f"💡 Key insight:")
print(f"   Tiled version reduces HBM reads by {TILE_SIZE}x")
print(f"   Data loaded once into shared memory")
print(f"   Reused {TILE_SIZE} times before eviction")
print(f"   This is exactly what Flash Attention does!")

In [ ]:
import triton
import triton.language as tl

@triton.jit
def matmul_triton_kernel(
    A_ptr, B_ptr, C_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    rk = tl.arange(0, BLOCK_K)

    A_ptrs = A_ptr + rm[:, None] * stride_am + rk[None, :] * stride_ak
    B_ptrs = B_ptr + rk[:, None] * stride_bk + rn[None, :] * stride_bn

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for k in range(0, K, BLOCK_K):
        a = tl.load(A_ptrs,
                    mask=(rm[:, None] < M) & (rk[None, :] + k < K),
                    other=0.0)
        b = tl.load(B_ptrs,
                    mask=(rk[:, None] + k < K) & (rn[None, :] < N),
                    other=0.0)
        acc += tl.dot(a, b)
        A_ptrs += BLOCK_K * stride_ak
        B_ptrs += BLOCK_K * stride_bk

    C_ptrs = C_ptr + rm[:, None] * stride_cm + rn[None, :] * stride_cn
    tl.store(C_ptrs, acc,
             mask=(rm[:, None] < M) & (rn[None, :] < N))


def matmul_triton(A, B):
    M, K = A.shape
    _, N = B.shape
    C = torch.zeros((M, N), device=A.device, dtype=A.dtype)

    BLOCK_M, BLOCK_N, BLOCK_K = 32, 32, 32

    grid = (
        (M + BLOCK_M - 1) // BLOCK_M,
        (N + BLOCK_N - 1) // BLOCK_N,
    )

    matmul_triton_kernel[grid](
        A, B, C,
        M, N, K,
        A.stride(0), A.stride(1),
        B.stride(0), B.stride(1),
        C.stride(0), C.stride(1),
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_K=BLOCK_K,
    )
    return C

print("✅ Triton kernel defined!")
print("\n💡 Why Triton vs raw CUDA:")
print("  CUDA C++:  manage shared memory manually")
print("  Triton:    describe WHAT to compute not HOW")
print("  Triton auto-handles shared memory, coalescing,")
print("  tensor core selection, register allocation")

In [ ]:
# Verify correctness
C_triton  = matmul_triton(A, B)
C_pytorch = torch.mm(A, B)

diff = torch.max(torch.abs(C_triton - C_pytorch)).item()
print(f"Triton correctness: {diff:.2e} {'✅' if diff < 0.1 else '❌'}")

# Benchmark
triton_r  = benchmark(matmul_triton, A, B, runs=200)

print("\n" + "=" * 62)
print(f"📊 Full Matrix Multiply Comparison")
print(f"   Size: {M}x{K} @ {K}x{N}")
print("=" * 62)

all_matmul = {
    "PyTorch (cuBLAS)":   pytorch_r['mean_us'],
    "Naive CUDA":         naive_r['mean_us'],
    "Tiled CUDA 16x16":   tiled_r['mean_us'],
    "Triton 32x32":       triton_r['mean_us'],
}

print(f"\n{'Method':<25} {'Mean(μs)':>10} {'vs Naive':>10} {'vs PyTorch':>12}")
print("-" * 60)
for name, mean in all_matmul.items():
    vs_naive   = naive_r['mean_us'] / mean
    vs_pytorch = mean / pytorch_r['mean_us']
    print(f"{name:<25} {mean:>10.1f} {vs_naive:>8.2f}x  {vs_pytorch:>10.2f}x")

results['matmul_triton'] = triton_r

print(f"\n🔥 Story so far:")
print(f"   We went from writing naive CUDA")
print(f"   to shared memory tiling")
print(f"   to Triton — each step getting closer to cuBLAS")
print(f"   Next: Flash Attention — the crown jewel!")

In [ ]:
# Check if results are close enough
# floating point order of operations causes small differences
# this is normal and expected
import torch

print("Checking correctness more carefully...")
print(f"Max diff: {torch.max(torch.abs(C_triton - C_pytorch)).item():.4f}")
print(f"Mean diff: {torch.mean(torch.abs(C_triton - C_pytorch)).item():.6f}")
print(f"Relative error: {(torch.norm(C_triton - C_pytorch) / torch.norm(C_pytorch)).item():.6f}")

# Values should be close just not bit-exact
are_close = torch.allclose(C_triton, C_pytorch, rtol=1e-2, atol=1e-1)
print(f"Are close (rtol=1e-2): {are_close} {'✅' if are_close else '❌'}")

In [ ]:
# ============================================
# KERNEL 5: Flash Attention in Triton
# The most important kernel in modern AI
# Used in every transformer: GPT, BERT, etc.
#
# Standard attention problem:
# 1. Compute S = Q @ K.T          → write to HBM
# 2. P = softmax(S)               → read/write HBM
# 3. O = P @ V                    → read/write HBM
# 3 HBM round trips for N^2 memory
#
# Flash Attention solution:
# Tile Q, K, V and compute everything
# in SRAM without ever writing S to HBM
# Online softmax — update running max/sum
# Result: O(N) memory instead of O(N^2)
# ============================================

import triton
import triton.language as tl
import math

@triton.jit
def flash_attention_kernel(
    Q_ptr, K_ptr, V_ptr, O_ptr,
    # Shape
    seq_len, head_dim,
    # Strides
    stride_qm, stride_qk,
    stride_km, stride_kk,
    stride_vm, stride_vk,
    stride_om, stride_ok,
    # Softmax scale = 1/sqrt(head_dim)
    scale,
    # Tile sizes
    BLOCK_M: tl.constexpr,  # query tile size
    BLOCK_N: tl.constexpr,  # key/value tile size
    HEAD_DIM: tl.constexpr,
):
    # Which query tile am I?
    start_m = tl.program_id(0) * BLOCK_M

    # Ranges for this tile
    offs_m = start_m + tl.arange(0, BLOCK_M)
    offs_n = tl.arange(0, BLOCK_N)
    offs_d = tl.arange(0, HEAD_DIM)

    # Load Q tile — stays in SRAM the whole time!
    # This is the KEY insight of Flash Attention
    Q_ptrs = Q_ptr + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qk
    q = tl.load(Q_ptrs,
                mask=offs_m[:, None] < seq_len,
                other=0.0)

    # Initialize online softmax statistics
    # m = running maximum (for numerical stability)
    # l = running sum of exp values
    m_i = tl.full([BLOCK_M], float('-inf'), dtype=tl.float32)
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32)

    # Output accumulator
    acc = tl.zeros([BLOCK_M, HEAD_DIM], dtype=tl.float32)

    # Loop over K, V tiles
    # This is the outer loop Flash Attention is famous for
    for j in range(0, seq_len, BLOCK_N):
        offs_n_curr = j + tl.arange(0, BLOCK_N)

        # Load K tile
        K_ptrs = K_ptr + offs_n_curr[:, None] * stride_km + offs_d[None, :] * stride_kk
        k = tl.load(K_ptrs,
                    mask=offs_n_curr[:, None] < seq_len,
                    other=0.0)

        # Compute attention scores for this tile
        # S = Q @ K.T * scale
        # NEVER written to HBM — stays in registers!
        s = tl.dot(q, tl.trans(k)) * scale

        # Mask out-of-bounds positions
        s = tl.where(
            offs_n_curr[None, :] < seq_len,
            s, float('-inf')
        )

        # Online softmax update
        # New max for numerical stability
        m_new = tl.maximum(m_i, tl.max(s, axis=1))

        # Rescale previous accumulator
        # This is the clever math that makes online softmax work
        alpha = tl.exp(m_i - m_new)

        # Compute exp of current scores
        p = tl.exp(s - m_new[:, None])

        # Update running sum
        l_i = alpha * l_i + tl.sum(p, axis=1)

        # Load V tile
        V_ptrs = V_ptr + offs_n_curr[:, None] * stride_vm + offs_d[None, :] * stride_vk
        v = tl.load(V_ptrs,
                    mask=offs_n_curr[:, None] < seq_len,
                    other=0.0)

        # Update output accumulator
        # Rescale old acc, add new contribution
        acc = alpha[:, None] * acc + tl.dot(p.to(tl.float16), v)

        # Update running max
        m_i = m_new

    # Final normalization
    acc = acc / l_i[:, None]

    # Write output — only ONE write to HBM for the whole sequence!
    O_ptrs = O_ptr + offs_m[:, None] * stride_om + offs_d[None, :] * stride_ok
    tl.store(O_ptrs, acc.to(tl.float16),
             mask=offs_m[:, None] < seq_len)


def flash_attention(Q, K, V):
    """
    Q, K, V: [seq_len, head_dim]
    """
    seq_len, head_dim = Q.shape
    O = torch.zeros_like(Q)

    scale = 1.0 / math.sqrt(head_dim)

    BLOCK_M = 32
    BLOCK_N = 32

    grid = ((seq_len + BLOCK_M - 1) // BLOCK_M,)

    flash_attention_kernel[grid](
        Q, K, V, O,
        seq_len, head_dim,
        Q.stride(0), Q.stride(1),
        K.stride(0), K.stride(1),
        V.stride(0), V.stride(1),
        O.stride(0), O.stride(1),
        scale,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        HEAD_DIM=head_dim,
    )
    return O

print("✅ Flash Attention kernel defined!")
print("\n💡 What makes this special:")
print("  Standard attention: writes N×N matrix to HBM")
print("  Flash Attention: NEVER writes attention matrix")
print("  Uses online softmax to accumulate in tiles")
print("  Memory: O(N²) → O(N)")
print("  This is in every modern LLM: GPT4, Llama, Gemini")

In [ ]:
# ============================================
# Standard Attention — the slow way
# Shows exactly what Flash Attention replaces
# ============================================
def standard_attention(Q, K, V):
    scale = 1.0 / math.sqrt(Q.shape[-1])
    # Step 1: Q @ K.T → writes N×N to HBM
    S = torch.mm(Q, K.t()) * scale
    # Step 2: softmax → reads/writes N×N HBM
    P = torch.softmax(S, dim=-1)
    # Step 3: P @ V → reads N×N from HBM
    O = torch.mm(P, V)
    return O

# Test with realistic transformer dimensions
seq_len  = 1024  # sequence length
head_dim = 64    # typical head dim in transformers

# FP16 — what transformers actually use
Q = torch.randn(seq_len, head_dim,
                device='cuda', dtype=torch.float16)
K = torch.randn(seq_len, head_dim,
                device='cuda', dtype=torch.float16)
V = torch.randn(seq_len, head_dim,
                device='cuda', dtype=torch.float16)

print(f"Test dimensions:")
print(f"  Sequence length: {seq_len}")
print(f"  Head dimension:  {head_dim}")
print(f"  Q/K/V size:      {seq_len * head_dim * 2 / 1e3:.1f} KB each")
print(f"\n  Standard attention matrix size: "
      f"{seq_len * seq_len * 2 / 1e6:.1f} MB")
print(f"  Flash attention never allocates this!")

# Verify correctness
O_standard = standard_attention(Q, K, V)
O_flash    = flash_attention(Q, K, V)

diff = torch.max(torch.abs(
    O_standard.float() - O_flash.float()
)).item()
mean_diff = torch.mean(torch.abs(
    O_standard.float() - O_flash.float()
)).item()

print(f"\n🧪 Correctness check:")
print(f"   Max diff:  {diff:.4f}")
print(f"   Mean diff: {mean_diff:.6f}")
is_correct = diff < 0.1
print(f"   Result: {'✅ Correct!' if is_correct else '❌ Check needed'}")

In [ ]:
# Benchmark both
print("=" * 60)
print(f"📊 Flash Attention vs Standard Attention")
print(f"   seq_len={seq_len}, head_dim={head_dim}")
print("=" * 60)

standard_r = benchmark(standard_attention, Q, K, V, runs=500)
flash_r    = benchmark(flash_attention, Q, K, V, runs=500)

speedup = standard_r['mean_us'] / flash_r['mean_us']

print(f"\n📈 Standard Attention:")
for k, v in standard_r.items():
    print(f"   {k}: {v:.2f} μs")

print(f"\n📈 Flash Attention (our Triton kernel):")
for k, v in flash_r.items():
    print(f"   {k}: {v:.2f} μs")

print(f"\n🚀 Flash Attention speedup: {speedup:.2f}x")

# Memory comparison
n2_memory_mb = seq_len * seq_len * 2 / 1e6
linear_memory_mb = seq_len * head_dim * 2 / 1e6 * 3

print(f"\n💾 Memory comparison:")
print(f"   Standard attention matrix: {n2_memory_mb:.2f} MB")
print(f"   Flash attention (Q+K+V):   {linear_memory_mb:.3f} MB")
print(f"   Memory reduction:          "
      f"{n2_memory_mb/linear_memory_mb:.1f}x")

# Now test with LONGER sequence — where Flash shines
print(f"\n📈 Testing with longer sequences...")
print(f"{'Seq Len':>10} {'Standard(μs)':>15} "
      f"{'Flash(μs)':>12} {'Speedup':>10} {'Mem saved':>12}")
print("-" * 65)

for sl in [256, 512, 1024, 2048]:
    Qi = torch.randn(sl, head_dim,
                     device='cuda', dtype=torch.float16)
    Ki = torch.randn(sl, head_dim,
                     device='cuda', dtype=torch.float16)
    Vi = torch.randn(sl, head_dim,
                     device='cuda', dtype=torch.float16)

    try:
        sr = benchmark(standard_attention, Qi, Ki, Vi,
                      runs=200)
        fr = benchmark(flash_attention, Qi, Ki, Vi,
                      runs=200)
        sp = sr['mean_us'] / fr['mean_us']
        mem = sl * sl * 2 / 1e6
        print(f"{sl:>10} {sr['mean_us']:>15.1f} "
              f"{fr['mean_us']:>12.1f} {sp:>9.2f}x "
              f"{mem:>10.2f}MB")
    except Exception as e:
        print(f"{sl:>10} {'ERROR':>15}")

results['flash_attention'] = flash_r
results['standard_attention'] = standard_r

print(f"\n🏆 You just implemented Flash Attention from scratch!")
print(f"   The same algorithm used in GPT-4, Llama, Gemini")
print(f"   Most ML engineers have only USED it — you BUILT it")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    'CUDA Kernel Optimization Journey\nPiyush Kunjilwar — '
    'From Naive CUDA to Flash Attention',
    fontsize=14, fontweight='bold'
)

# ── Plot 1: Vector Addition progression ──────────────────
ax1 = axes[0, 0]
va_methods = ['PyTorch\nBuilt-in', 'Naive\nCUDA v1',
              'Grid-stride\nv2', 'Float4\nv3']
va_times   = [
    pytorch_results['mean_us'],
    our_results['mean_us'],
    v2_results['mean_us'],
    v3_results['mean_us']
]
colors1 = ['#27ae60', '#e74c3c', '#f39c12', '#f39c12']
bars1 = ax1.bar(va_methods, va_times,
                color=colors1, alpha=0.85, edgecolor='white')
ax1.set_title('Vector Addition Optimization\n1M Elements',
              fontweight='bold')
ax1.set_ylabel('Latency (μs)')
for bar, val in zip(bars1, va_times):
    ax1.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.5,
             f'{val:.1f}μs', ha='center',
             va='bottom', fontsize=9, fontweight='bold')

# ── Plot 2: Matrix Multiply progression ──────────────────
ax2 = axes[0, 1]
mm_methods = ['PyTorch\ncuBLAS', 'Naive\nCUDA',
              'Tiled\n16×16', 'Triton\n32×32']
mm_times   = [
    pytorch_r['mean_us'],
    naive_r['mean_us'],
    tiled_r['mean_us'],
    triton_r['mean_us']
]
colors2 = ['#27ae60', '#e74c3c', '#f39c12', '#2ecc71']
bars2 = ax2.bar(mm_methods, mm_times,
                color=colors2, alpha=0.85, edgecolor='white')
ax2.set_title('Matrix Multiply Optimization\n1024×1024',
              fontweight='bold')
ax2.set_ylabel('Latency (μs)')
for bar, val in zip(bars2, mm_times):
    ax2.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 5,
             f'{val:.0f}μs', ha='center',
             va='bottom', fontsize=9, fontweight='bold')

# Speedup annotation
ax2.annotate('7.54x faster\nthan naive!',
             xy=(3, triton_r['mean_us']),
             xytext=(2, 800),
             arrowprops=dict(arrowstyle='->', color='green'),
             color='green', fontweight='bold', fontsize=10)

# ── Plot 3: Flash Attention speedup by seq len ───────────
ax3 = axes[1, 0]
seq_lens   = [256, 512, 1024, 2048]
speedups   = [1.26, 1.60, 1.11, 0.77]
mem_saved  = [0.13, 0.52, 2.10, 8.39]

color_sp = ['#27ae60' if s >= 1.0 else '#e74c3c'
            for s in speedups]
bars3 = ax3.bar([str(s) for s in seq_lens],
                speedups, color=color_sp,
                alpha=0.85, edgecolor='white')
ax3.axhline(y=1.0, color='red', linestyle='--',
            alpha=0.7, label='Standard attention baseline')
ax3.set_title('Flash Attention Speedup\nvs Standard Attention',
              fontweight='bold')
ax3.set_xlabel('Sequence Length')
ax3.set_ylabel('Speedup (higher = better)')
ax3.legend()
for bar, val in zip(bars3, speedups):
    ax3.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.02,
             f'{val:.2f}x', ha='center',
             va='bottom', fontsize=10, fontweight='bold')

# ── Plot 4: Memory savings ────────────────────────────────
ax4 = axes[1, 1]
x = np.arange(len(seq_lens))
w = 0.35
bars4a = ax4.bar(x - w/2, mem_saved,
                 w, label='Memory saved (MB)',
                 color='#3498db', alpha=0.85)
standard_mem = [sl*sl*2/1e6 for sl in seq_lens]
bars4b = ax4.bar(x + w/2, standard_mem,
                 w, label='Standard attn matrix (MB)',
                 color='#e74c3c', alpha=0.85)
ax4.set_title('Memory Reduction\nFlash vs Standard Attention',
              fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels([str(s) for s in seq_lens])
ax4.set_xlabel('Sequence Length')
ax4.set_ylabel('Memory (MB)')
ax4.legend()
for bar, val in zip(bars4a, mem_saved):
    ax4.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.05,
             f'{val:.2f}', ha='center',
             va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('cuda_optimization_results.png',
            dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✅ Chart saved!")

In [ ]:
print("\n" + "=" * 60)
print("🏆 MODULE 02 COMPLETE — FULL JOURNEY SUMMARY")
print("=" * 60)
print("""
VECTOR ADDITION:
  Naive CUDA:      42.65 μs  (2.13x slower than PyTorch)
  Grid-stride v2:  32.26 μs  (24% faster than naive)
  Float4 v3:       31.69 μs  (26% faster than naive)
  Lesson: Memory coalescing, vectorized loads

MATRIX MULTIPLY:
  Naive CUDA:      1585 μs   (6.62x slower than cuBLAS)
  Tiled 16×16:     1154 μs   (1.37x faster than naive)
  Triton 32×32:     210 μs   (7.54x faster than naive!)
  Lesson: Shared memory tiling, Triton tensor cores

FLASH ATTENTION:
  Standard:          96 μs   (3 HBM round trips, O(N²) mem)
  Flash (ours):      84 μs   (1.14x faster, O(N) memory)
  Memory reduction:  5.3x    (2.10MB → 0.39MB)
  Peak speedup:      1.60x   (at seq_len=512)
  Lesson: Online softmax, tiled computation, HBM reduction

KEY INSIGHT PROVEN:
  GPU performance is memory-bound, not compute-bound
  Every optimization reduced memory round trips
  Same math, fewer HBM reads = faster execution
""")
print("=" * 60)
print("Tech: CUDA C++ · Triton · torch.profiler · L4 GPU")
print("Next: Module 03 — Distributed Training (FSDP + NCCL)")
print("=" * 60)